In [2]:
# =============================================================================
# INDIVIDUAL METHOD RUNNER
# Quick testing of a single method on a single dataset
# =============================================================================

# -----------------------------------------------------------------------------
# CONFIGURATION - CHANGE THESE VALUES
# -----------------------------------------------------------------------------

METHOD = "node"           # Method to run (e.g., 'xgboost', 'catboost', 'tabpfn', 'mlp')
DATASET = "0004.base_model"        # Dataset name (e.g., '0014.hmeq', '0001.gmsc')
TASK = "lgd"                  # Task type: 'pd' (classification) or 'lgd' (regression)

# -----------------------------------------------------------------------------
# FIXED SETTINGS (for quick testing)
# -----------------------------------------------------------------------------

ROW_LIMIT = 10000             # Limit rows for fast execution
MAX_EPOCHS = 15              # Max epochs for deep learning methods
CV_SPLITS = 1                # Single fold
TUNE = False                 # No HPO
SEED = 42                    # Random seed
TEST_SIZE = 0.2              # Test set fraction
VAL_SIZE = 0.2               # Validation set fraction

# -----------------------------------------------------------------------------
# SETUP
# -----------------------------------------------------------------------------

import sys
from pathlib import Path
import pickle
import json
from datetime import datetime

# Add project root to path (notebook is in notebooks/ folder)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"\n{'='*60}")
print(f" Running: {METHOD} on {DATASET} ({TASK.upper()})")
print(f"{'='*60}")
print(f"  Row limit:  {ROW_LIMIT}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  CV splits:  {CV_SPLITS}")
print(f"  HPO:        {TUNE}")
print(f"{'='*60}\n")

# -----------------------------------------------------------------------------
# RUN METHOD
# -----------------------------------------------------------------------------

from src.methods.method_runner import run_talent_method, get_available_methods

# Show available methods
available = get_available_methods()
print(f"Available classical methods: {available['classical']}")
print(f"Available deep methods: {available['deep'][:10]}... ({len(available['deep'])} total)")
print()

# Run the method
results = run_talent_method(
    task=TASK,
    dataset=DATASET,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    cv_splits=CV_SPLITS,
    seed=SEED,
    row_limit=ROW_LIMIT,
    method=METHOD,
    max_epoch=MAX_EPOCHS,
    tune=TUNE,
    verbose=True,
)

# -----------------------------------------------------------------------------
# DISPLAY RESULTS
# -----------------------------------------------------------------------------

print(f"\n{'='*60}")
print(f" RESULTS")
print(f"{'='*60}")

for fold_id, fold_results in results.items():
    print(f"\nFold {fold_id}:")
    print(f"  Train time: {fold_results['train_time']:.2f}s")
    print(f"  Samples:    {len(fold_results['y_true'])}")
    
    if TASK == 'lgd':
        print(f"  Clipped:    {fold_results['n_clipped_below']} below, {fold_results['n_clipped_above']} above")
    
    print(f"\n  Metrics:")
    for metric_name, metric_value in fold_results['metrics'].items():
        if not (isinstance(metric_value, float) and metric_value != metric_value):  # Skip NaN
            print(f"    {metric_name:20s}: {metric_value:.4f}")

# -----------------------------------------------------------------------------
# SAVE RESULTS
# -----------------------------------------------------------------------------

# Create output directory
output_dir = PROJECT_ROOT / 'results' / 'individual_method_runner'
output_dir.mkdir(parents=True, exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"{METHOD}_{DATASET}_{TASK}_{timestamp}"

# Save as pickle (full results)
pickle_path = output_dir / f"{filename}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to: {pickle_path}")

# Save summary as JSON (metrics only, for easy viewing)
summary = {
    'method': METHOD,
    'dataset': DATASET,
    'task': TASK,
    'timestamp': timestamp,
    'config': {
        'row_limit': ROW_LIMIT,
        'max_epochs': MAX_EPOCHS,
        'cv_splits': CV_SPLITS,
        'tune': TUNE,
        'seed': SEED,
    },
    'folds': {}
}

for fold_id, fold_results in results.items():
    summary['folds'][fold_id] = {
        'train_time': fold_results['train_time'],
        'n_samples': len(fold_results['y_true']),
        'metrics': {k: v for k, v in fold_results['metrics'].items() if not (isinstance(v, float) and v != v)},
    }
    if TASK == 'lgd':
        summary['folds'][fold_id]['n_clipped_below'] = fold_results['n_clipped_below']
        summary['folds'][fold_id]['n_clipped_above'] = fold_results['n_clipped_above']

json_path = output_dir / f"{filename}.json"
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {json_path}")

print(f"\n{'='*60}")
print(f" DONE")
print(f"{'='*60}")

Project root: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit

 Running: node on 0004.base_model (LGD)
  Row limit:  10000
  Max epochs: 15
  CV splits:  1
  HPO:        False

Available classical methods: ['LinearRegression', 'LogReg', 'NCM', 'NaiveBayes', 'RandomForest', 'catboost', 'dummy', 'knn', 'lightgbm', 'svm', 'xgboost']
Available deep methods: ['amformer', 'autoint', 'bishop', 'danets', 'dcn2', 'dnnr', 'excelformer', 'ftt', 'grande', 'grownet']... (38 total)


Running node (deep) on 0004.base_model (LGD)

Preparing data with 1 CV splits...
Fold IDs: [1]
First fold ID: 1

Directory setup:
  Config directory (persistent): C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\config_hpo\lgd\0004.base_model
  Checkpoint directory (temp):   C:\Users\U0152019\AppData\Local\Temp\talent_ckpt_0004.base_model_node_wx1po_4h

[HPO] Mode: DISABLED
[HPO] All folds: Will use TALENT's default hyperparameters

Fold 1/1
using 

1it [00:00,  7.68it/s]

epoch 0, val, loss=0.8329 regression result=0.3168
Epoch: 0, Time cost: 1.7788889408111572


epoch 1, train 1/1, loss=0.9986 lr=0.001
best epoch 0, best val res=0.3168


1it [00:00, 13.49it/s]

...

epoch 14, train 1/1, loss=0.9843 lr=0.001
best epoch 13, best val res=0.3153


1it [00:00, 11.94it/s]


epoch 14, val, loss=0.8259 regression result=0.3152
Epoch: 14, Time cost: 1.9691405296325684
best epoch 14, best val res=0.3152


1it [00:00, 10.61it/s]

Test: loss=0.9579
[MAE]=0.3381
[R2]=0.0089
[RMSE]=0.3749

Fold 1 metrics:
  R2: 0.0089
  MSE: 0.1405
  RMSE: 0.3749
  MAE: 0.3381
  MedAE: 0.3116
  MaxError: 0.6675
  Explained_Variance: 0.0101
  MAPE: 608.7532
  Pearson_Corr: 0.4967
  Spearman_Corr: 0.4204

Completed 1 folds for node


 RESULTS

Fold 1:
  Train time: 32.63s
  Samples:    149
  Clipped:    0 below, 0 above

  Metrics:
    R2                  : 0.0089
    MSE                 : 0.1405
    RMSE                : 0.3749
    MAE                 : 0.3381
    MedAE               : 0.3116
    MaxError            : 0.6675
    Explained_Variance  : 0.0101
    MAPE                : 608.7532
    Pearson_Corr        : 0.4967
    Spearman_Corr       : 0.4204

Results saved to: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\individual_method_runner\node_0004.base_model_lgd_20251216_165649.pkl
Summary saved to: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCr